# Musanga V4 — NB04 V4.003: safe MS inventory

**No raster pixel reads anywhere in this notebook.** Following the Colab out-of-memory failure, this version only inspects file headers and survey bounding boxes intersected with the study ROI. This provides approximate *maximum geographic coverage*, not measured valid pixel coverage or spectral quality.

It writes exclusively to `Outputs/v4/04_ms_qa_v4_003` and does not overwrite previous results.

After a fresh runtime restart, run all code cells. We will sample small native-resolution *windows*, carefully selected from the ROI, in the next phase only after reviewing block layouts.


In [ ]:
# 00 — Reuse NB00 inventory and frozen ROI; metadata only, no image read
from pathlib import Path
import json, math, re, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
NB00=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/04_ms_qa_v4_003'
OUT.mkdir(parents=True,exist_ok=True)
REG=NB00/'00_ms_registry.json'
HEADERS=NB00/'00_ms_raster_headers.csv'
OVERLAP=NB00/'00_ms_rgb_extent_audit.csv'
assert REG.is_file() and HEADERS.is_file() and OVERLAP.is_file(),(
    'First run NB00 MS inventory cells; missing',REG,HEADERS,OVERLAP)
ms_manifest=json.loads(REG.read_text())
ms=pd.read_csv(HEADERS).fillna({'error':''})
overlap=pd.read_csv(OVERLAP)
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
roi=gpd.read_file(ROI_PATH)
assert roi.crs is not None and len(roi)>0
RGB_REFERENCE=ms_manifest['rgb_reference']
RGB_DIR=DATA/'Ortho_RGB'
RGB_FILES={group:RGB_DIR/name for group,name in RGB_REFERENCE.items()}
assert all(p.is_file() for p in RGB_FILES.values()),RGB_FILES
EDGE_DISTANCE_M=(1,3,5,10,20,30,50)
print('Available surveys:', sorted(ms.group.dropna().unique()))
print('NB00 MS records:',len(ms),'| paired extent records:',len(overlap))
display(ms[['group','band_hint','name','bands','dtype','nodata','error']].head(40))


In [ ]:
# 01 — Band headers only, ZERO source pixel reads
rows=[]
for rec in ms.itertuples():
    if getattr(rec,'error',''):continue
    path=Path(rec.path)
    if not path.is_file():continue
    with rasterio.Env(GDAL_CACHEMAX=32):
        with rasterio.open(path) as ds:
            for b in range(1,ds.count+1):
                rows.append(dict(group=rec.group,file=path.name,path=str(path),
                    band=b,band_hint=rec.band_hint,
                    description=ds.descriptions[b-1],
                    colorinterp=str(ds.colorinterp[b-1]),
                    dtype=ds.dtypes[b-1],scale=ds.scales[b-1],offset=ds.offsets[b-1],
                    nodata=ds.nodatavals[b-1],crs=str(ds.crs),
                    width=ds.width,height=ds.height,
                    pixel_x_m=abs(ds.transform.a),pixel_y_m=abs(ds.transform.e),
                    block_shape=str(ds.block_shapes[b-1]),
                    pixel_distribution='NOT SAMPLED',file_bytes=path.stat().st_size))
    print('HEADERS:',rec.group,rec.name,flush=True)
BAND_AUDIT=pd.DataFrame(rows)
assert not BAND_AUDIT.empty
BAND_AUDIT.to_csv(OUT/'04_ms_band_metadata_only.csv',index=False)
display(BAND_AUDIT[['group','file','band','band_hint','description','dtype',
                    'scale','offset','nodata','pixel_x_m','block_shape']])
print('No pixel data read, no band-value statistics inferred.')


In [ ]:
# 02 — ROI / RGB / MS BOUNDING-BOX overlap ONLY, ZERO pixel reads
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.warp import transform_bounds
summary=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        study=unary_union(roi.to_crs(rgb.crs).geometry).intersection(box(*rgb.bounds))
        area=float(study.area)
        if area<=0:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            with rasterio.Env(GDAL_CACHEMAX=32):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    bb=box(*transform_bounds(src.crs,rgb.crs,*src.bounds,densify_pts=21))
                    frac=float(study.intersection(bb).area/area)
                    summary.append(dict(group=group,file=rec.name,band_hint=rec.band_hint,
                        nominal_bands=int(src.count),roi_bbox_overlap_fraction=frac,
                        rgb_roi_area_ha=area/1e4,valid_pixel_coverage='NOT TESTED',
                        registration='NOT TESTED'))
            print('BBOX:',group,rec.name,round(frac,3),flush=True)
SUMMARY=pd.DataFrame(summary)
assert not SUMMARY.empty
SUMMARY.to_csv(OUT/'04_ms_rgb_roi_bbox_only.csv',index=False)
display(SUMMARY.sort_values(['group','file']).round(3))
print('Bounding-box overlap is only an upper limit on valid MS pixel coverage.')


In [ ]:
# 03 — Metadata coverage figure, NO pixel reads
d=SUMMARY.sort_values(['group','file'])
fig,ax=plt.subplots(figsize=(9,max(4,len(d)*.23)))
ax.barh(np.arange(len(d)),d.roi_bbox_overlap_fraction.to_numpy())
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels([f"{r.group}: {r.file[:40]}" for r in d.itertuples()],fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0,1.02)
ax.set_xlabel('Bounding-box overlap with RGB ∩ ROI (not actual valid coverage)')
ax.set_title('MS inventory — metadata only')
fig.tight_layout()
fig.savefig(OUT/'04_ms_bbox_metadata_summary.png',dpi=120,bbox_inches='tight')
plt.show()
plt.close(fig)
print('DONE:',OUT)


## Next stage

Share the band metadata table, including `block_shape`, and ROI bbox summary first. We will then create a **separate**, strictly bounded **small-native-window** pixel QA approach with a hard per-window pixel limit and limited GDAL caching. The current notebook must not be interpreted as proof that an MS band is valid or radiometrically calibrated.
